In [1]:
!pip show torch

Name: torch
Version: 2.6.0+cu124
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org/
Author: PyTorch Team
Author-email: packages@pytorch.org
License: BSD-3-Clause
Location: /usr/local/lib/python3.11/dist-packages
Requires: filelock, fsspec, jinja2, networkx, nvidia-cublas-cu12, nvidia-cuda-cupti-cu12, nvidia-cuda-nvrtc-cu12, nvidia-cuda-runtime-cu12, nvidia-cudnn-cu12, nvidia-cufft-cu12, nvidia-curand-cu12, nvidia-cusolver-cu12, nvidia-cusparse-cu12, nvidia-cusparselt-cu12, nvidia-nccl-cu12, nvidia-nvjitlink-cu12, nvidia-nvtx-cu12, sympy, triton, typing-extensions
Required-by: accelerate, fastai, peft, sentence-transformers, timm, torchaudio, torchdata, torchvision


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Core Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from scipy import stats
from scipy.stats import randint, uniform

## IMPORT & EXPLORE

In [4]:
from sentence_transformers import SentenceTransformer

In [5]:
import torch

In [6]:
labse_transformer = SentenceTransformer('sentence-transformers/LaBSE')

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/461 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/114 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/2.36M [00:00<?, ?B/s]

2_Dense/pytorch_model.bin:   0%|          | 0.00/2.36M [00:00<?, ?B/s]

In [7]:
!unzip "/content/drive/MyDrive/flip_vlm/flip_data_vlm.zip" -d "/content/flip_data_vlm"

Выходные данные были обрезаны до нескольких последних строк (5000).
  inflating: /content/flip_data_vlm/flip_data_vlm/category_3/images/Вытяжка_KHI_9931_W_4512.jpg  
  inflating: /content/flip_data_vlm/flip_data_vlm/category_3/images/Вытяжка_KHI_9931_X_4975.jpg  
  inflating: /content/flip_data_vlm/flip_data_vlm/category_3/images/Вытяжка_KHI_9997_GW_4910.jpg  
  inflating: /content/flip_data_vlm/flip_data_vlm/category_3/images/Вытяжка_KHI_9997_X_4914.jpg  
  inflating: /content/flip_data_vlm/flip_data_vlm/category_3/images/Вытяжка_KHP_5637_GNX_4927.jpg  
  inflating: /content/flip_data_vlm/flip_data_vlm/category_3/images/Вытяжка_KHP_6501_GN_4939.jpg  
  inflating: /content/flip_data_vlm/flip_data_vlm/category_3/images/Вытяжка_KHP_6617_GN_4928.jpg  
  inflating: /content/flip_data_vlm/flip_data_vlm/category_3/images/Вытяжка_KHP_6617_GW_4972.jpg  
  inflating: /content/flip_data_vlm/flip_data_vlm/category_3/images/Вытяжка_KHP_6637_GNX_4970.jpg  
  inflating: /content/flip_data_vlm/flip_d

In [8]:
all_products_combined = pd.read_csv("/content/flip_data_vlm/flip_data_vlm/all_products_combined.csv")

## CLEAN & PREPARE

In [9]:
all_products_combined = all_products_combined.dropna()

In [10]:
from pathlib import Path

def is_valid_image_path(path):
    return Path(path).is_file() and not Path(path).is_dir()

all_products_combined['colab_image_path'] = (
    all_products_combined['local_image_path']
    .fillna("")
    .astype(str)
    .apply(lambda x: x.replace("/root/flip/data/", "/content/flip_data_vlm/flip_data_vlm/"))
)

# Filter only valid paths
all_products_combined = all_products_combined[
    all_products_combined['colab_image_path'].apply(is_valid_image_path)
]

In [11]:
all_products_combined[['colab_image_path','title']].sample(10)

,colab_image_path,title
10575,/content/flip_data_vlm/flip_data_vlm/category_...,"Подсвечник «Гармония», белый"
10286,/content/flip_data_vlm/flip_data_vlm/category_...,Электрогирлянда «Роса»
14736,/content/flip_data_vlm/flip_data_vlm/category_...,Салфетки влажные для пластиковых поверхностей ...
439,/content/flip_data_vlm/flip_data_vlm/category_...,"Ведро складное «Флекс», морская волна"
10068,/content/flip_data_vlm/flip_data_vlm/category_...,"Диффузор ароматический «Local honey», мед"
4199,/content/flip_data_vlm/flip_data_vlm/category_...,"Бутылка для воды складная, зеленый"
4333,/content/flip_data_vlm/flip_data_vlm/category_...,Топор-колун с клиновидным полотном
1483,/content/flip_data_vlm/flip_data_vlm/category_...,Зевник
10142,/content/flip_data_vlm/flip_data_vlm/category_...,Лампа настольная
2044,/content/flip_data_vlm/flip_data_vlm/category_...,"Юбка-шорты женская, голубой, 44RU/S"


In [12]:
all_products_combined = all_products_combined[
    all_products_combined["colab_image_path"] != '/content/flip_data_vlm/flip_data_vlm/'
]

In [16]:
image_title_pairs = list(all_products_combined[['colab_image_path','title']].sample(frac=1, random_state=42, replace=False).itertuples(index=False, name=None))

In [17]:
len(image_title_pairs)

18079

In [18]:
image_title_pairs

[('/content/flip_data_vlm/flip_data_vlm/category_3/images/Отпариватель_ручной_221.jpg',
  'Отпариватель ручной'),
 ('/content/flip_data_vlm/flip_data_vlm/category_1/images/Набор_блоков_для_йоги_салатовый_3223.jpg',
  'Набор блоков для йоги, салатовый'),
 ('/content/flip_data_vlm/flip_data_vlm/category_2/images/Щетка_для_уборки_3711.jpg',
  'Щетка для уборки'),
 ('/content/flip_data_vlm/flip_data_vlm/category_1/images/Мангал_разборный_с_ребрами_жесткости_с_кулинарной__4862.jpg',
  'Мангал разборный с ребрами жесткости с кулинарной книгой'),
 ('/content/flip_data_vlm/flip_data_vlm/category_1/images/Набор_ложек_2127.jpg',
  'Набор ложек'),
 ('/content/flip_data_vlm/flip_data_vlm/category_3/images/Отпариватель_AUL7788_26.jpg',
  'Отпариватель AUL7788'),
 ('/content/flip_data_vlm/flip_data_vlm/category_3/images/Пылесос_вертикальный_IZ300EU_2035.jpg',
  'Пылесос вертикальный IZ300EU'),
 ('/content/flip_data_vlm/flip_data_vlm/category_3/images/Мясорубка_BR1607_2890.jpg',
  'Мясорубка BR1607')

##### DATASET

In [19]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T

In [20]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [21]:
torch.cuda.get_device_properties()

_CudaDeviceProperties(name='Tesla T4', major=7, minor=5, total_memory=15095MB, multi_processor_count=40, uuid=b220b902-7d32-835e-4ee8-83d16f1ded55, L2_cache_size=4MB)

In [22]:
class ImageTitleDataset(Dataset):
    def __init__(self, data, transform=None):
        self.data = data  # list of (image_path, text)
        self.transform = transform or T.Compose([
            T.Resize((224, 224)),
            T.ToTensor(),
            T.Normalize([0.485, 0.456, 0.406],
                        [0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path, text = self.data[idx]
        image = Image.open(img_path).convert("RGB")
        image = self.transform(image)
        return image, text

In [23]:
image_title18k = ImageTitleDataset(image_title_pairs)

In [34]:
print(image_title18k.__len__())
print(image_title18k.__getitem__(42))

18079
(tensor([[[2.2489, 2.2489, 2.2489,  ..., 2.2489, 2.2489, 2.2489],
         [2.2489, 2.2489, 2.2489,  ..., 2.2489, 2.2489, 2.2489],
         [2.2489, 2.2489, 2.2489,  ..., 2.2489, 2.2489, 2.2489],
         ...,
         [2.2489, 2.2489, 2.2489,  ..., 2.2489, 2.2489, 2.2489],
         [2.2489, 2.2489, 2.2489,  ..., 2.2489, 2.2489, 2.2489],
         [2.2489, 2.2489, 2.2489,  ..., 2.2489, 2.2489, 2.2489]],

        [[2.4286, 2.4286, 2.4286,  ..., 2.4286, 2.4286, 2.4286],
         [2.4286, 2.4286, 2.4286,  ..., 2.4286, 2.4286, 2.4286],
         [2.4286, 2.4286, 2.4286,  ..., 2.4286, 2.4286, 2.4286],
         ...,
         [2.4286, 2.4286, 2.4286,  ..., 2.4286, 2.4286, 2.4286],
         [2.4286, 2.4286, 2.4286,  ..., 2.4286, 2.4286, 2.4286],
         [2.4286, 2.4286, 2.4286,  ..., 2.4286, 2.4286, 2.4286]],

        [[2.6400, 2.6400, 2.6400,  ..., 2.6400, 2.6400, 2.6400],
         [2.6400, 2.6400, 2.6400,  ..., 2.6400, 2.6400, 2.6400],
         [2.6400, 2.6400, 2.6400,  ..., 2.6400, 2.6

##### DATA LOADER

In [25]:
image_title18k_dataloader32 = DataLoader( image_title18k, batch_size = 32, shuffle = True  )

In [26]:
image_title18k_dataloader32

## MODEL BUILD

#### MODEL ARCHITECTURE

In [27]:
import torchvision

In [28]:
class FlipDualEncoderv2(torch.nn.Module):
  def __init__(self, embedding_dim = 256):
    super().__init__()

    #ResNet50 as visual encoder
    resnet50 = torchvision.models.resnet50(pretrained = True)
    resnet50.fc = torch.nn.Identity()
    self.visual_encoder = resnet50
    self.image_projection = torch.nn.Linear(2048, embedding_dim)

    #LaBSE as text encoder
    self.text_encoder = SentenceTransformer('sentence-transformers/LaBSE')
    self.text_projection = torch.nn.Linear(768, embedding_dim)

  def forward(self, images, texts):

    # Image embedding
    image_encoded = self.visual_encoder(images)
    image_embedding = torch.nn.functional.normalize(self.image_projection(image_encoded), p=2, dim=-1)

    # Text embedding
    with torch.no_grad():  # freeze LaBSE initially (optional)
        text_features = self.text_encoder.encode(
            texts,
            convert_to_tensor=True,
            normalize_embeddings=False
        )  # (B, 768)

    text_embedding = self.text_projection(text_features)  # (B, 256)
    text_embedding = torch.nn.functional.normalize(text_embedding, p=2, dim=-1)

    return image_embedding, text_embedding



In [29]:
flip_dual_encoderv2 = FlipDualEncoderv2().to(device)


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:01<00:00, 98.9MB/s]


In [30]:
total_params = sum(p.numel() for p in flip_dual_encoderv2.parameters())
trainable_params = sum(p.numel() for p in flip_dual_encoderv2.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Total parameters: 495,746,880
Trainable parameters: 495,746,880


#### LOSS

In [31]:
def cosine_similarity_loss(image_embeddings, text_embeddings, temperature=0.07):
    """
    image_embeddings: (B, D)
    text_embeddings:  (B, D)
    """
    # Cosine similarity matrix: (B, B)
    similarity_matrix = torch.matmul(image_embeddings, text_embeddings.T) / temperature

    # Labels: diagonal is positive
    targets = torch.arange(similarity_matrix.size(0)).to(similarity_matrix.device)

    # Symmetric loss: image-to-text + text-to-image
    loss_i2t = torch.nn.functional.cross_entropy(similarity_matrix, targets)
    loss_t2i = torch.nn.functional.cross_entropy(similarity_matrix.T, targets)

    return (loss_i2t + loss_t2i) / 2

#### MODEL TRAINING

In [32]:
adam_optimizer = torch.optim.Adam( flip_dual_encoderv2.parameters(), lr = 1e-4 )

In [36]:
loss_history = []
early_stop_loss = 0.004
early_stopped = False  # flag to track if we early stopped

for epoch in range(300):
    print(f"Epoch {epoch+1} started")

    for step, (images, texts) in enumerate(image_title18k_dataloader32):
        print(f"  Step {step} | Batch size: {len(images)}")

        images = images.to(device)

        adam_optimizer.zero_grad()

        # Forward pass
        image_embeddings, text_embeddings = flip_dual_encoderv2(images, texts)

        # Compute contrastive loss
        loss = cosine_similarity_loss(image_embeddings, text_embeddings)

        # Backpropagation
        loss.backward()
        adam_optimizer.step()
        print(f"Step: {step} cosine_similarity_loss: {loss.item():.6f}")

        loss_history.append(loss.item())

        # Check early stopping condition
        if loss.item() <= early_stop_loss:
            print(f"\n🚀 Early stopping triggered at epoch {epoch+1}, step {step+1} with loss {loss.item():.6f}")

            # Save safely
            flip_dual_encoderv2_cpu = flip_dual_encoderv2.to('cpu')
            torch.save(flip_dual_encoderv2_cpu.state_dict(), "/content/drive/MyDrive/flip_vlm/early_stopped_model.pt")
            print("✅ Model weights saved as 'early_stopped_model.pt'")

            # Move back to device
            flip_dual_encoderv2.to(device)

            early_stopped = True
            break  # break inner loop

    print(f"Epoch: {epoch+1} cosine_similarity_loss: {loss.item():.6f}")

    if early_stopped:
        break  # break outer loop

print("🎉 Training finished. Last recorded loss:", loss_history[-1])

Epoch 1 started
  Step 0 | Batch size: 32
Step: 0 cosine_similarity_loss: 0.076315
  Step 1 | Batch size: 32
Step: 1 cosine_similarity_loss: 0.071479
  Step 2 | Batch size: 32
Step: 2 cosine_similarity_loss: 0.058928
  Step 3 | Batch size: 32
Step: 3 cosine_similarity_loss: 0.051920
  Step 4 | Batch size: 32
Step: 4 cosine_similarity_loss: 0.085332
  Step 5 | Batch size: 32
Step: 5 cosine_similarity_loss: 0.050836
  Step 6 | Batch size: 32
Step: 6 cosine_similarity_loss: 0.079163
  Step 7 | Batch size: 32
Step: 7 cosine_similarity_loss: 0.083442
  Step 8 | Batch size: 32
Step: 8 cosine_similarity_loss: 0.153595
  Step 9 | Batch size: 32
Step: 9 cosine_similarity_loss: 0.054720
  Step 10 | Batch size: 32
Step: 10 cosine_similarity_loss: 0.058522
  Step 11 | Batch size: 32
Step: 11 cosine_similarity_loss: 0.094402
  Step 12 | Batch size: 32
Step: 12 cosine_similarity_loss: 0.115252
  Step 13 | Batch size: 32
Step: 13 cosine_similarity_loss: 0.087030
  Step 14 | Batch size: 32
Step: 14 co

KeyboardInterrupt: 